<a href="https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:**

A page is worth reviewing if its click-through rate is meaningfully below what's
typical for pages at a similar search position, and it has enough impressions to
matter — because a low CTR on a page nobody sees isn't worth anyone's time, but the
same low CTR on a well-seen page is a real, fixable opportunity.

**Reason codes this rule can output:**

- `low_ctr_for_position_with_volume` — the page's CTR is at or below the median CTR
  for its position bucket, AND it has at least 100 impressions. This is currently the
  only reason code the rule produces, since only one condition combination is being
  scored in this version.

(A future, richer version of this rule could output additional reason codes — e.g.
`severely_underperforming` for pages far below their position's expected CTR, or
`insufficient_volume` for pages that are low-CTR but too low-traffic to prioritize —
but this baseline intentionally stays to one clear, defensible rule.)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "pandas"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb, pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
    ORDER BY content_hash_id
""").df()

df["ctr"] = df["total_clicks"] / df["total_impressions"]
print("Rows:", len(df))
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738


,content_hash_id,avg_position,total_impressions,total_clicks,ctr
0,content_000005d4ced12088,72.854861,86.0,0.0,0.000000
1,content_00007bd2985b77c3,5.269565,47.0,0.0,0.000000
2,content_0000cd28fbda69f3,4.251282,29.0,0.0,0.000000
3,content_0000d495bfbfb4a8,3.333333,15.0,0.0,0.000000
4,content_00014efc121d911d,4.964683,116.0,1.0,0.008621


In [2]:
import pandas as pd

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

signal1 = df.groupby("position_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    avg_ctr=("ctr", "mean")
).reset_index()

print(signal1)

  position_bucket      n   avg_ctr
0             1-3  16144  0.010589
1            4-10  81988  0.004926
2           11-20  32203  0.003211
3           21-50  33288  0.002287
4             50+  11681  0.000903


**Signal 1 — CTR vs. position: CONFIRMED**

Average CTR decreases monotonically as position gets worse (0.0106 at position 1-3,
down to 0.0009 at position 50+), across 176,738 pages with real sample sizes in every
bucket (n ranges from 11,681 to 81,988). This directly confirms the assumption behind
the CTR-fix flag from the session: position and CTR are genuinely linked, so a rule
comparing a page's CTR to its position peers is measuring something real.

In [3]:
median_ctr_by_bucket = df.groupby("position_bucket", observed=True)["ctr"].transform("median")
df["low_ctr_for_position"] = df["ctr"] <= median_ctr_by_bucket

df["volume_bucket"] = pd.cut(
    df["total_impressions"],
    bins=[0, 100, 500, 2000, 1000000],
    labels=["low", "medium", "high", "very_high"]
)

signal2 = df.groupby("volume_bucket", observed=True).agg(
    n=("content_hash_id", "count"),
    pct_low_ctr=("low_ctr_for_position", "mean")
).reset_index()

print(signal2)

  volume_bucket      n  pct_low_ctr
0           low  75506     0.912749
1        medium  39356     0.681802
2          high  32012     0.292921
3     very_high  29864     0.046745


**Signal 2 — Volume: MIXED**

Volume is strongly related to CTR performance, but not the way "quick-win = high volume"
might suggest. Only 4.7% of very-high-volume pages are low-CTR-for-their-position, versus
91.3% of low-volume pages. This makes sense — high-traffic pages tend to already be
well-optimized, since poor performance there would be highly visible. The real opportunity
pool sits in the low/medium volume buckets (75,506 + 39,356 = 114,862 pages), where most
pages are underperforming for their position but haven't been scrutinized. This means the
rule should weight toward "enough volume to matter, but not already-optimized," not simply
"more volume = better."

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# Only count pages with "enough but not excessive" volume as candidates,
# then rank by how impression-worthy they are within that band —
# avoids both extremes: too little volume to matter, or so much volume
# the page is likely already optimized (per Signal 2)
in_target_volume_band = (df["total_impressions"] >= 100) & (df["total_impressions"] <= 5000)
underperforming = (df["ctr"] <= median_ctr_by_bucket).astype(int)

df["score"] = underperforming * in_target_volume_band.astype(int) * df["total_impressions"]

df["reason_code"] = "low_ctr_for_position_with_volume"
df["action"] = "review_metadata"

queue = df.sort_values("score", ascending=False).reset_index(drop=True)

print(queue[["content_hash_id", "avg_position", "ctr", "total_impressions", "score", "reason_code", "action"]].head(20))

             content_hash_id  avg_position  ctr  total_impressions   score  \
0   content_15bb18400a219f5d     32.765669  0.0             5000.0  5000.0   
1   content_877402d4c7c8f084      8.128211  0.0             4999.0  4999.0   
2   content_835af1106517866a      4.840309  0.0             4997.0  4997.0   
3   content_2495437c921d02bb     10.256878  0.0             4995.0  4995.0   
4   content_0262ad57029b3b4a     31.940979  0.0             4994.0  4994.0   
5   content_6b4a87382da50de5      7.154498  0.0             4968.0  4968.0   
6   content_6e83801fd04299a5     26.412017  0.0             4954.0  4954.0   
7   content_c69ec04764885fe5     32.018451  0.0             4949.0  4949.0   
8   content_1d9f00db9317d029     23.314451  0.0             4938.0  4938.0   
9   content_09bfcfef1c478130      5.717108  0.0             4937.0  4937.0   
10  content_3a025922b92c2084     31.758329  0.0             4922.0  4922.0   
11  content_b34ce20454d991f8      6.366944  0.0             4904

In [5]:
import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Saved:", len(queue), "rows to work/outputs/baseline_action_score.csv")

Saved: 176738 rows to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
print(queue[["content_hash_id", "avg_position", "ctr", "total_impressions"]].head(20).to_string())

             content_hash_id  avg_position  ctr  total_impressions
0   content_15bb18400a219f5d     32.765669  0.0             5000.0
1   content_877402d4c7c8f084      8.128211  0.0             4999.0
2   content_835af1106517866a      4.840309  0.0             4997.0
3   content_2495437c921d02bb     10.256878  0.0             4995.0
4   content_0262ad57029b3b4a     31.940979  0.0             4994.0
5   content_6b4a87382da50de5      7.154498  0.0             4968.0
6   content_6e83801fd04299a5     26.412017  0.0             4954.0
7   content_c69ec04764885fe5     32.018451  0.0             4949.0
8   content_1d9f00db9317d029     23.314451  0.0             4938.0
9   content_09bfcfef1c478130      5.717108  0.0             4937.0
10  content_3a025922b92c2084     31.758329  0.0             4922.0
11  content_b34ce20454d991f8      6.366944  0.0             4904.0
12  content_f7a26a41bb14e6db      4.995875  0.0             4896.0
13  content_d3a8876d618d8ecd      9.839464  0.0             48

All rows share: action = review_metadata, reason_code = low_ctr_for_position_with_volume.
Scoring was updated after reviewing Signal 2: instead of ranking by raw impressions (which
favored only the highest-traffic pages), the rule now restricts to a target volume band
(100–5,000 impressions) — "enough volume to matter, not so much it's likely already
optimized" — then ranks by impressions within that band. Confidence varies by position,
since Signal 1 showed CTR naturally drops as position worsens.

1. content_15bb18400a219f5d — pos 32.8, 5,000 impr, 0 clicks. Confidence: LOW-MEDIUM
   (mid-low position, some CTR drop expected). Wrong if: typical for this position range.
2. content_877402d4c7c8f084 — pos 8.1, 4,999 impr, 0 clicks. Confidence: HIGH (strong
   position, zero clicks is a real anomaly). Wrong if: query-intent mismatch, not metadata.
3. content_835af1106517866a — pos 4.8, 4,997 impr, 0 clicks. Confidence: HIGH (excellent
   position). Wrong if: tracking/measurement issue rather than a real content problem.
4. content_2495437c921d02bb — pos 10.3, 4,995 impr, 0 clicks. Confidence: HIGH.
   Wrong if: recent content change still stabilizing.
5. content_0262ad57029b3b4a — pos 31.9, 4,994 impr, 0 clicks. Confidence: LOW-MEDIUM.
   Wrong if: expected outcome for this position.
6. content_6b4a87382da50de5 — pos 7.2, 4,968 impr, 0 clicks. Confidence: HIGH.
   Wrong if: snippet already fine, real issue is elsewhere.
7. content_6e83801fd04299a5 — pos 26.4, 4,954 impr, 0 clicks. Confidence: MEDIUM.
   Wrong if: consistent with typical mid-position performance.
8. content_c69ec04764885fe5 — pos 32.0, 4,949 impr, 0 clicks. Confidence: LOW-MEDIUM.
   Wrong if: unremarkable for this position.
9. content_1df00db9317d029 — pos 23.3, 4,938 impr, 0 clicks. Confidence: MEDIUM.
   Wrong if: within normal range per Signal 1.
10. content_09bfcfef1c478130 — pos 5.7, 4,937 impr, 0 clicks. Confidence: HIGH.
    Wrong if: very new page, short observation window.
11. content_3a025922b92c2084 — pos 15.8, 4,922 impr, 0 clicks. Confidence: MEDIUM-HIGH.
    Wrong if: borderline position, may be normal variance.
12. content_b34ce20454d991f8 — pos 6.4, 4,904 impr, 0 clicks. Confidence: HIGH.
    Wrong if: measurement gap rather than genuine underperformance.
13. content_f7a26a41bb14e6db — pos 5.0, 4,896 impr, 0 clicks. Confidence: HIGH (near-top
    position). Wrong if: page targets a low-intent query with naturally low CTR.
14. content_d3a8876d618d8ecd — pos 9.8, 4,896 impr, 0 clicks. Confidence: HIGH.
    Wrong if: recent change, still stabilizing.
15. content_f8978c1bac069ce0 — pos 38.3, 4,888 impr, 0 clicks. Confidence: LOW.
    Wrong if: expected for this position range.
16. content_4eb86d20c2e05ff8 — pos 21.6, 4,886 impr, 0 clicks. Confidence: MEDIUM.
    Wrong if: consistent with Signal 1's mid-position baseline.
17. content_62963a3d337e30e6 — pos 58.9, 4,881 impr, 0 clicks. Confidence: LOW.
    Wrong if: very low CTR is expected this far down.
18. content_1dca7a9d2ab1e5a3 — pos 8.0, 4,876 impr, 0 clicks. Confidence: HIGH.
    Wrong if: title/snippet already fine, different root cause.
19. content_0fec78df64213e11 — pos 50.3, 4,872 impr, 0 clicks. Confidence: LOW.
    Wrong if: not fixable by metadata at this position.
20. content_ea6db3ecfe821513 — pos 33.8, 4,868 impr, 0 clicks. Confidence: LOW-MEDIUM.
    Wrong if: typical for this position range.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks from my top 20**

Even with the improved rule (volume band 100–5,000, instead of raw impressions),
several picks are still weak on inspection:

- Rows 1, 5, 7, 8, 11, 15, 16, 17, 19, 20 (positions 26–59): these fall into the
  same trap as before, just less severely — CTR naturally drops toward zero at
  poor positions (per Signal 1), so a zero-click page at position 30+ isn't
  necessarily a real anomaly, just what's expected. My LOW confidence tags on
  these rows reflect that directly.

- The rule still treats "below the position-bucket median" as a binary flag, not
  a matter of degree. A page at position 5 with zero clicks and a page at position
  50 with zero clicks are scored identically if their impressions are similar,
  even though the first is a shocking anomaly and the second is close to expected.

- A stronger future version would weight the score by how far below the position
  bucket's expected CTR a page falls (not just below/above), so genuine anomalies
  at strong positions would clearly outrank routine low performance at weak
  positions — this is a real limitation I'm keeping visible rather than hiding.

**Leakage check:** No product decision flags or future-window data were used as inputs.
All features (avg_position, total_impressions, ctr) were computed only from month=2026-03,
filtered to gsc_data_available IS TRUE — never touching the sealed final month
(June 2026 / the _sample table). The label itself (low_ctr_for_position) is derived
from ctr, which is why total_clicks/ctr are used only to build the label and score,
never fed back in as a separate "feature" alongside the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
  (e.g. Signal 2 was marked MIXED rather than forcing a clean CONFIRMED/OPPOSITE;
  confidence notes in the top-20 use LOW/MEDIUM/HIGH rather than certainty language)
- [x] Committed to my repo under work/notebooks/